# 02 · End-to-end BM25 + document generation (current approach)

Runs the full pipeline on one PDF and reports evaluation metrics:

`ingest (text + OCR fallback)` → `paragraph_merge chunking` → `per-field BM25 retrieval` → `dedupe context` → `structured IntakeForm generation (Ollama)` → `metrics`.

**Prerequisites**
- `pip install -r requirements.txt`
- A PDF in `data/documents/` (samples in `~/rag-testing/legal_documents`).
- `.env` with `OLLAMA_BASE_URL` (default `http://192.168.1.74:1143`) and `OLLAMA_MODEL` set; the host must be reachable for the generation cell.

Only `paragraph_merge` and the `common/` contracts come from the package; BM25, generation, and metrics are implemented inline here so this notebook doubles as a reference for filling in the stage stubs.

In [1]:
# Make the code under src/ importable when running from notebooks/
import sys, os, re, json
sys.path.insert(0, os.path.abspath("../src"))
from pathlib import Path

from chunking import CHUNKERS
from common.config import settings
from common.schema import IntakeForm, INTAKE_FIELDS
from common.types import ScoredChunk, RetrievalResult

print("ollama:", settings.ollama_base_url, "| model:", settings.ollama_model or "(unset)")
print("intake fields:", INTAKE_FIELDS)

ollama: http://192.168.1.74:11434 | model: gemma4:e2b
intake fields: ['incident-description', 'incident-when', 'incident-where', 'incident-damages', 'desired-outcome', 'prior-actions']


## 1. Ingest — PDF → text (OCR fallback per page)

In [2]:
!pip install pdf2image pytesseract

In [2]:
def extract_text(pdf_path, use_ocr=True):
    """Read the text layer with pypdf; OCR any page that has no extractable text."""
    from pypdf import PdfReader

    reader = PdfReader(str(pdf_path))
    pages = []
    for i, page in enumerate(reader.pages):
        t = (page.extract_text() or "").strip()
        if not t and use_ocr:
            t = _ocr_page(pdf_path, i)
        pages.append(t)
    return "\n\n".join(p for p in pages if p)


def _ocr_page(pdf_path, page_index):
    """OCR a single page. Requires poppler (pdf2image) + tesseract (pytesseract)."""
    try:
        from pdf2image import convert_from_path
        import pytesseract

        imgs = convert_from_path(str(pdf_path), first_page=page_index + 1, last_page=page_index + 1)
        return "\n".join(pytesseract.image_to_string(im) for im in imgs).strip()
    except Exception as e:
        print(f"  [ocr skipped page {page_index}: {e}]")
        return ""


DOC_DIR = Path("../data/documents")
pdfs = sorted(DOC_DIR.glob("*.pdf"))
assert pdfs, f"drop a PDF into {DOC_DIR.resolve()} (samples in ~/rag-testing/legal_documents)"

pdf_path = pdfs[0]
text = extract_text(pdf_path)
print(pdf_path.name, "->", len(text), "chars")
print(text)

20201023- RELEASE ORDER.pdf -> 1395 chars
Republic of the Philippines
MUNICIPAL TRIAL COURT
_ Third Judicial Region
Sta. Rosa, Niueva Ecija

~oO0-

PEOPLE OF THE PHILIPPINES,
Plaintiff
-versus- : . CRIM. CASES NOS. 09-20-6850 to
se 09-20-6882

‘MAYNARD R. CRUZ and
EDMUNDO BASALLO,
Xisrern enna nn nnn name nnnrrnnn nnn nnannnnnn nnn mma nenn X

RELEASE ORDER

Accused, MAYNARD _R. CRUZ and EDMUNDO BASALLO,
through their respective relatives posted cash bond in the amount of
THREE THOUSAND ‘PESOS in each case or NINETY-NINE
THOUSAND) PESOS for each accused, and for the thirty-three (33)
counts of cases for their provisional liberty, which amount were
deposited witit this Court under Official Receipts No. 7939568. and
7939569, respectively all dated October 23, 2020, and finding the same
to be sufficient i amount, forms and substance, the same are, as it
were, hereby APPROVED.

WHEREFORE, the immediate release of MAYNARD R. CRUZ
and EDMUNDO BASAL.LO from custody of the Talavera Police Stat

## 2. Chunk — `paragraph_merge` (from the package)

In [3]:
chunks = CHUNKERS.get("paragraph_merge")(text, source=pdf_path.stem, chunk_min=1000, chunk_max=1500)
print(len(chunks), "chunks")
for c in chunks:
    print(c.metadata, "| len", len(c.page_content), "|", c.page_content)

13 chunks
{'source': '20201023- RELEASE ORDER', 'chunk_index': 0} | len 97 | Republic of the Philippines MUNICIPAL TRIAL COURT _ Third Judicial Region Sta. Rosa, Niueva Ecija
{'source': '20201023- RELEASE ORDER', 'chunk_index': 1} | len 5 | ~oO0-
{'source': '20201023- RELEASE ORDER', 'chunk_index': 2} | len 94 | PEOPLE OF THE PHILIPPINES, Plaintiff -versus- : . CRIM. CASES NOS. 09-20-6850 to se 09-20-6882
{'source': '20201023- RELEASE ORDER', 'chunk_index': 3} | len 100 | ‘MAYNARD R. CRUZ and EDMUNDO BASALLO, Xisrern enna nn nnn name nnnrrnnn nnn nnannnnnn nnn mma nenn X
{'source': '20201023- RELEASE ORDER', 'chunk_index': 4} | len 13 | RELEASE ORDER
{'source': '20201023- RELEASE ORDER', 'chunk_index': 5} | len 524 | Accused, MAYNARD _R. CRUZ and EDMUNDO BASALLO, through their respective relatives posted cash bond in the amount of THREE THOUSAND ‘PESOS in each case or NINETY-NINE THOUSAND) PESOS for each accused, and for the thirty-three (33) counts of cases for their provisional liber

## 3. Per-field BM25 retrieval

Each intake field gets its own query ("individual user response" framing). We use `rank_bm25` directly so we can surface real scores — `BM25Retriever` hides them.

> BM25 scores are *relative* and can be ≤ 0 on small corpora (IDF penalizes terms common to many chunks). We therefore rank by top-k rather than hard-filtering at zero; `MIN_SCORE` is an optional cutoff.

In [4]:
FIELD_QUERIES = {
    "incident-description": "On October 10, 2020, I was accused of robbing a convenience store located at 789 A. Cruz Street, Talavera. The police arrested me and I was charged with 33 counts of robbery and related offenses. I posted a bond of 99,000 pesos and was released on October 23, 2020. I am now preparing for my arraignment on November 25, 2020.",
    "incident-when": "October 10, 2020",
    "incident-where": "Convenience store, 789 A. Cruz Street, Talavera, Nueva Ecija",
    "incident-damages": "The store owner claims loss of goods worth 200,000 pesos, plus legal fees.",
    "desired-outcome": "I want to be acquitted or have the charges reduced.",
    "prior-actions": "Arrested, posted bond, released by court, arraignment scheduled."
}

In [ ]:
def tokenize(s):
    return re.findall(r"[a-z0-9]+", s.lower())


from rank_bm25 import BM25Okapi

corpus_tokens = [tokenize(c.page_content) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)


def query_bm25(query, k=None, min_score=None):
    k = k or settings.top_k
    scores = bm25.get_scores(tokenize(query))
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    scored = [
        ScoredChunk(
            chunk_index=chunks[i].metadata["chunk_index"],
            score=float(scores[i]),
            text=chunks[i].page_content,
            metadata=chunks[i].metadata,
        )
        for i in order
        if min_score is None or scores[i] >= min_score
    ]
    return RetrievalResult(query=query, scored_chunks=scored)



per_field = {field: query_bm25(q, k=3, min_score=3) for field, q in FIELD_QUERIES.items()}


### Dedupe retrieved chunks across fields → generation context

In [36]:
def dedupe_context(per_field):
    """Collapse per-field hits into unique chunks, recording which fields matched.

    Mirrors the rag_context array in ~/rag-testing/tfidf_retrieval_results.json.
    Kept for the debug artifact and retrieval metrics.
    """
    by_index = {}
    for field, res in per_field.items():
        for sc in res.scored_chunks:
            e = by_index.get(sc.chunk_index)
            if e is None:
                e = {"chunk_index": sc.chunk_index, "chunk_text": sc.text,
                     "best_score": sc.score, "matched_fields": []}
                by_index[sc.chunk_index] = e
            e["best_score"] = max(e["best_score"], sc.score)
            e["matched_fields"].append(field)
    return sorted(by_index.values(), key=lambda e: e["best_score"], reverse=True)


def build_intake_prompts(field_queries):
    """Lay the per-field answers out under labeled fields, mirroring the
    "Incident:" section of buildPromptsString() in VisionDrafter
    (src/pages/ai-thinking/AiThinkingPage.tsx). Driven by FIELD_QUERIES, the
    synthetic client responses."""
    lines = [
        "Incident:",
        f"    What: {field_queries.get('incident-description', '')}",
        f"    When: {field_queries.get('incident-when', '')}",
        f"    Where: {field_queries.get('incident-where', '')}",
        f"    Damages Involved: {field_queries.get('incident-damages', '')}",
        f"    Desired Outcome: {field_queries.get('desired-outcome', '')}",
        f"    Prior Actions: {field_queries.get('prior-actions', '')}",
    ]
    return "\n".join(lines)


def build_context_block(context_chunks):
    """The deduped BM25 hits, as supporting evidence from the source document."""
    if not context_chunks:
        return "(no supporting context retrieved)"
    return "\n\n".join(f"[chunk {c['chunk_index']}] {c['chunk_text']}" for c in context_chunks)


# Unique chunks (debug artifact + retrieval metrics).
context_chunks = dedupe_context(per_field)

# Intake = labeled client answers, followed by the retrieved source context.
intake_prompts = build_intake_prompts(FIELD_QUERIES)
context = (
    f"{intake_prompts}\n\n"
    "Supporting context from the source document:\n"
    f"{build_context_block(context_chunks)}"
)

print(len(context_chunks), "unique chunks |", len(context), "prompt chars")


4 unique chunks | 1733 prompt chars


In [57]:
print(intake_prompts + "\n\n")

for a in context_chunks:
    print("chunk_index:", str(a["chunk_index"]), "\ntext: ", a["chunk_text"], "\n")


print()

for c in chunks:
    print(c.metadata, "| len", len(c.page_content), "|", c.page_content, "\n")


Incident:
    What: On October 10, 2020, I was accused of robbing a convenience store located at 789 A. Cruz Street, Talavera. The police arrested me and I was charged with 33 counts of robbery and related offenses. I posted a bond of 99,000 pesos and was released on October 23, 2020. I am now preparing for my arraignment on November 25, 2020.
    When: October 10, 2020
    Where: Convenience store, 789 A. Cruz Street, Talavera, Nueva Ecija
    Damages Involved: The store owner claims loss of goods worth 200,000 pesos, plus legal fees.
    Desired Outcome: I want to be acquitted or have the charges reduced.
    Prior Actions: Arrested, posted bond, released by court, arraignment scheduled.


chunk_index: 7 
text:  Meantime, the. arraignment of the accused and Pre-trial Conference of these cases are hereby set on November 25, 2020 at 

chunk_index: 5 
text:  Accused, MAYNARD _R. CRUZ and EDMUNDO BASALLO, through their respective relatives posted cash bond in the amount of THREE THOUSAND

## 4. Document generation — structured `IntakeForm` via Ollama

In [39]:
from typing import List

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# Port of VisionDrafter's document_generation (src-tauri/src/document_generation/mod.rs):
# generate a lawyer-reviewable DRAFT LEGAL DOCUMENT, not an intake form. The model
# picks the document type from the facts and returns Quill-compatible HTML.


class MissingInfoPlaceholder(BaseModel):
    placeholder: str = Field(description="e.g. [PLACEHOLDER]")
    needed_information: str = Field(description="What information must be supplied")


class SuggestedAnnex(BaseModel):
    annex_label: str = Field(description="e.g. Annex A")
    description: str = Field(description="Description of the evidence/document")


class LetterDraft(BaseModel):
    """Mirrors LetterDraft / LETTER_DRAFT_JSON_SCHEMA in mod.rs."""

    selected_document_type: str = Field(description="The document type chosen")
    suggested_annexes: List[SuggestedAnnex] = Field(default_factory=list)
    draft_document: str = Field(description="Quill-compatible HTML legal document draft")
    missing_information_placeholders: List[MissingInfoPlaceholder] = Field(default_factory=list)
    legal_basis_status: str = Field(
        description="provided | partially_provided | not_provided | needs_lawyer_verification"
    )


# LETTER_SYSTEM from mod.rs
LETTER_SYSTEM = (
    "You are a legal drafting assistant for a Philippine law office. "
    "Always output valid JSON only."
)


def build_letter_draft_prompt(intake_prompts: str) -> str:
    """Port of build_letter_draft_prompt() in mod.rs.

    `intake_prompts` is the labeled client answers plus the retrieved source
    context (the `context` string from build_intake_prompts + build_context_block).
    The follow-up answers step is intentionally omitted in this notebook.
    """
    return (
        "You are a legal drafting assistant for a Philippine law office.\n"
        "Generate a lawyer-reviewable draft legal document based on the intake data.\n"
        "Intake information:\n"
        f"{intake_prompts}\n"
        "Rules:\n"
        "- Determine the most appropriate document type based on the facts and Philippine procedural rules.\n"
        "- Use formal Philippine legal drafting style.\n"
        "- Do not fabricate or assume facts, legal citations, or procedural details.\n"
        "- Use placeholders like [PLACEHOLDER] for missing but necessary information about the narrative or format.\n"
        "- Use placeholders when referring to either the Name or Address of the Client, Adversary, or Witness "
        "([Client Name], [Client Address], [Adversary1 Name], [Adversary1 Address], [Witness1 Name], [Witness1 Address] etc.)\n"
        "- Mention supporting evidence as annexes when applicable.\n"
        "- Avoid emotional or exaggerated language.\n"
        "- Use English.\n"
        "- Output JSON only.\n"
        "- `draft_document` must contain Quill-compatible HTML only (e.g. <p>, <strong>, <ol>, <ul>, <br>) "
        "without Markdown or <html>/<body> tags.\n"
    )


assert settings.ollama_model, "set OLLAMA_MODEL in .env"
llm = ChatOllama(
    base_url=settings.ollama_base_url,
    model=settings.ollama_model,
    temperature=settings.temperature,
    num_ctx=settings.num_ctx,
)
structured = llm.with_structured_output(LetterDraft)

# `context` = labeled client answers + the retrieved source context (built above).
# No follow-up step in this notebook.
draft = structured.invoke(
    [SystemMessage(content=LETTER_SYSTEM), HumanMessage(content=build_letter_draft_prompt(context))]
)
generated = draft.model_dump()

print("document type:", generated["selected_document_type"])
print("legal basis  :", generated["legal_basis_status"])
print("annexes      :", len(generated["suggested_annexes"]), "| placeholders:", len(generated["missing_information_placeholders"]))
print("\n--- draft_document (HTML) ---\n")
print(generated["draft_document"])


document type: Motion for Consideration of Mitigation of Sentence/Charges
legal basis  : Rules of Court, Revised Penal Code, and relevant procedural rules concerning bail and criminal procedure in the Philippines.
annexes      : 2 | placeholders: 0

--- draft_document (HTML) ---

<p><strong>REPUBLIC OF THE PHILIPPINES</strong></p><p><strong>REGIONAL TRIAL COURT</strong></p><p><strong>[BRANCH NUMBER]</strong></p><p><strong>[CITY/MUNICIPALITY]</strong></p><br><br><strong>[CASE NAME/CRIMINAL CASE NO.]</strong><br><br><strong>IN THE MATTER OF:</strong><br><br><strong>[ACCUSED NAME],</strong><br>Accused</p><br><br><strong>MOTION FOR CONSIDERATION OF MITIGATION OF CHARGES AND/OR SENTENCE</strong><br><br>COMES NOW the Accused, [ACCUSED NAME], by and through undersigned counsel, and respectfully moves this Honorable Court to consider the following facts and circumstances in his/her favor and prays for the following relief:<br><br>WHEREAS, the Accused was arrested and charged with 33 counts of 

## 5. Evaluation metrics

No gold labels yet, so these are **intrinsic** metrics:

- **Retrieval** — best/mean BM25 score per field, field coverage, context size.
- **Generation** — field completeness, and a *grounding* proxy: fraction of each generated field's tokens that appear in the retrieved context (low grounding ⇒ likely hallucination).

Set `GOLD` to a reference intake dict (alias keys) to additionally get per-field exact-match and token-Jaccard against ground truth.

In [58]:
def token_overlap(value, context_text):
    """Fraction of value tokens present in the context (grounding proxy)."""
    vt = set(tokenize(value))
    return len(vt & set(tokenize(context_text))) / len(vt) if vt else 0.0


def jaccard(a, b):
    ta, tb = set(tokenize(a)), set(tokenize(b))
    return len(ta & tb) / len(ta | tb) if (ta or tb) else 1.0


# --- retrieval metrics ---
best_scores = {f: (r.scored_chunks[0].score if r.scored_chunks else 0.0) for f, r in per_field.items()}
fields_with_hits = sum(1 for r in per_field.values() if r.scored_chunks)
retrieval_metrics = {
    "best_score_per_field": {f: round(s, 4) for f, s in best_scores.items()},
    "mean_best_score": round(sum(best_scores.values()) / len(best_scores), 4),
    "field_coverage": round(fields_with_hits / len(per_field), 3),
    "unique_context_chunks": len(context_chunks),
    "context_chunk_ratio": round(len(context_chunks) / max(1, len(chunks)), 3),
}

# --- generation metrics (text intake fields only) ---
text_fields = [f for f in INTAKE_FIELDS]
present = {f: bool(str(generated.get(f, "")).strip()) for f in text_fields}
grounding = {f: round(token_overlap(str(generated.get(f, "")), context), 3) for f in text_fields}
generation_metrics = {
    "field_completeness": round(sum(present.values()) / len(text_fields), 3),
    "empty_fields": [f for f, ok in present.items() if not ok],
    "grounding_per_field": grounding,
    "mean_grounding": round(sum(grounding.values()) / len(grounding), 3),
    "n_adversaries": len(generated.get("adversaries", [])),
    "n_witnesses": len(generated.get("witnesses", [])),
}

# --- optional gold comparison ---
GOLD = None  # e.g. json.load(open("../data/gold/<file>.json"))["synthetic_intake"]
gold_metrics = None
if GOLD:
    exact = {f: float(str(generated.get(f, "")).strip().lower() == str(GOLD.get(f, "")).strip().lower()) for f in text_fields}
    jac = {f: round(jaccard(str(generated.get(f, "")), str(GOLD.get(f, ""))), 3) for f in text_fields}
    gold_metrics = {
        "exact_match": {f: exact[f] for f in text_fields},
        "mean_exact_match": round(sum(exact.values()) / len(exact), 3),
        "token_jaccard": jac,
        "mean_token_jaccard": round(sum(jac.values()) / len(jac), 3),
    }

metrics = {"retrieval": retrieval_metrics, "generation": generation_metrics, "gold": gold_metrics}
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "retrieval": {
    "best_score_per_field": {
      "incident-description": 20.1563,
      "incident-when": 3.0738,
      "incident-where": 3.953,
      "incident-damages": 0.0,
      "desired-outcome": 0.0,
      "prior-actions": 0.0
    },
    "mean_best_score": 4.5305,
    "field_coverage": 0.5,
    "unique_context_chunks": 4,
    "context_chunk_ratio": 0.308
  },
  "generation": {
    "field_completeness": 0.0,
    "empty_fields": [
      "incident-description",
      "incident-when",
      "incident-where",
      "incident-damages",
      "desired-outcome",
      "prior-actions"
    ],
    "grounding_per_field": {
      "incident-description": 0.0,
      "incident-when": 0.0,
      "incident-where": 0.0,
      "incident-damages": 0.0,
      "desired-outcome": 0.0,
      "prior-actions": 0.0
    },
    "mean_grounding": 0.0,
    "n_adversaries": 0,
    "n_witnesses": 0
  },
  "gold": null
}


## 6. Save artifacts

In [ ]:
result = {
    "source_file": pdf_path.name,
    "total_chunks": len(chunks),
    "retrieval_debug": {
        field: {
            "query_text": res.query,
            "best_score": (res.scored_chunks[0].score if res.scored_chunks else 0.0),
            "top_chunks": [vars(sc) for sc in res.scored_chunks],
        }
        for field, res in per_field.items()
    },
    "rag_context": context_chunks,
    "generated_intake": generated,
    "metrics": metrics,
}

out_path = Path("../data") / f"02_eval_e2e_{pdf_path.stem}.results.json"
out_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print("wrote", out_path)